# MoBot: find the ArUco marker and dock

Runs the full robot (estimator + planner + controller in `src/`) in the TurtleBot3 dock room, draws the path, scores it, and records a video.

**The room:** 6 m x 4 m with three boxes, a sofa (brown box along the north wall) and a round coffee table in front of it. The charging dock with the ArUco marker is on the east wall. The robot only uses its own sensors (camera, LiDAR, wheel encoders, gyro).

Run the cells in order (or **Runtime → Run all**). Works on a CPU or GPU runtime.

In [ ]:
import os, shutil
if not os.path.exists("Mobile-robot"):
    !git clone -q https://github.com/anne-sharon/Mobile-robot.git
%cd Mobile-robot
!pip install -q mujoco opencv-python matplotlib

# Pick an offscreen renderer: EGL on a GPU runtime, software OSMesa on a CPU runtime.
if shutil.which("nvidia-smi"):
    icd = "/usr/share/glvnd/egl_vendor.d/10_nvidia.json"
    if not os.path.exists(icd):
        os.makedirs(os.path.dirname(icd), exist_ok=True)
        with open(icd, "w") as f:
            f.write('{"file_format_version": "1.0.0", "ICD": {"library_path": "libEGL_nvidia.so.0"}}')
    os.environ["MUJOCO_GL"] = "egl"
else:
    !apt-get -qq update
    !apt-get -qq install -y --fix-missing libosmesa6-dev > /dev/null
    os.environ["MUJOCO_GL"] = "osmesa"
os.environ["PYOPENGL_PLATFORM"] = os.environ["MUJOCO_GL"]
print("renderer:", os.environ["MUJOCO_GL"])

## 1. One episode, with the path drawn on the overhead view
Colours show what the planner was doing: search (spin), explore, approach, align, dock.

In [ ]:
from eval.plot_episode import run_and_plot
result = run_and_plot()
print(result)

## 2. Score it the official way
Uses `eval/run_episode.py` (the judge's ground truth is only used for scoring).

In [ ]:
from eval.run_episode import run_episode
print(run_episode())

## 3. Video of the robot docking
Left: overhead view with the path drawn as the robot goes. Right: the robot's camera, with the ArUco tag outlined in green once detected. Takes about a minute. To keep the video, open the **Files** panel, go into `Mobile-robot`, right-click `docking.mp4` and choose **Download**.

In [ ]:
!pip install -q imageio-ffmpeg
import cv2, numpy as np, imageio.v2 as imageio
from IPython.display import Video, display
from src.world import Sim, PARK_POSE
from src.contracts import CONTROL_DT
from src.estimator import Estimator
from src.planner import Planner
from src.control import Controller
from src.judge import Judge
from src.marker import detect_marker
from eval.run_episode import sense

TOP_W, TOP_H = 720, 480
COLOURS = dict(search=(150, 150, 150), explore=(224, 160, 48), approach=(48, 112, 208),
               align=(160, 64, 192), dock=(48, 160, 80), done=(48, 160, 80))

def make_video(out="docking.mp4", max_time=120.0, every=2, fps=30):
    sim = Sim()
    cal = sim.sensors.calibration
    est, planner, ctrl, judge = Estimator(cal), Planner(cal), Controller(cal), Judge(sim)
    last, tick, path, frames = dict(lidar=None, image=None), 0, [], []
    gx, gy = judge.world_to_px(*PARK_POSE[:2], TOP_W, TOP_H)

    while sim.time < max_time:
        obs, last = sense(sim, tick, last)
        pose = est.update(obs)
        plan = planner.update(pose, obs, est.dock)
        v, w = ctrl.update(pose, plan)
        x, y, _ = judge.true_pose()
        path.append((*judge.world_to_px(x, y, TOP_W, TOP_H), plan.mode))

        if tick % every == 0 or plan.done:
            # left: overhead view with the path so far
            top = judge.render_topdown(TOP_W, TOP_H)
            for (x0, y0, _), (x1, y1, m) in zip(path, path[1:]):
                cv2.line(top, (int(x0), int(y0)), (int(x1), int(y1)), COLOURS.get(m, (0, 0, 0)), 3)
            cv2.drawMarker(top, (int(gx), int(gy)), (220, 30, 30), cv2.MARKER_CROSS, 18, 2)
            # right: the robot's camera, with the tag outlined when detected
            cam = obs["image"].copy()
            res = detect_marker(cam, cal["camera_matrix"], cal["marker"])
            if res is not None:
                cv2.polylines(cam, [res[0].astype(np.int32)], True, (0, 220, 0), 3)
            frame = np.hstack([top, cam])
            cv2.rectangle(frame, (0, 0), (frame.shape[1], 36), (46, 8, 18), -1)
            cv2.putText(frame, f"t = {sim.time:4.1f} s    stage: {plan.mode}", (12, 25),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.8, (246, 238, 233), 2)
            frames.append(frame)

        if plan.done:
            break
        sim.drive(v, w, CONTROL_DT)
        tick += 1

    # hold the last frame for 2 seconds with the result on it
    err = judge.park_error()
    end = frames[-1].copy()
    msg = f"PARKED  -  error {err['pos_m']*100:.1f} cm, {err['heading_deg']:.1f} deg" if err["parked"] else "NOT PARKED"
    cv2.rectangle(end, (0, end.shape[0] - 50), (end.shape[1], end.shape[0]), (46, 8, 18), -1)
    cv2.putText(end, msg, (12, end.shape[0] - 16), cv2.FONT_HERSHEY_SIMPLEX, 0.9, (246, 238, 233), 2)
    frames += [end] * (2 * fps)

    imageio.mimsave(out, frames, fps=fps, codec="libx264", quality=7, macro_block_size=8)
    return out, err

out, result = make_video()
print(result)
display(Video(out, embed=True, width=900))